# Resampling / Validation Statistics Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Resampling / Validation Statistics**.  
It demonstrates practical validation workflows used in machine learning, surrogate modeling, and scientific data analysis.

Topics covered:

1. Train / validation / test split  
2. Holdout validation  
3. k-fold cross-validation  
4. Stratified cross-validation  
5. Leave-One-Out cross-validation (LOOCV)  
6. Repeated random subsampling  
7. Bootstrap resampling  
8. Out-of-bag intuition  
9. Permutation validation  
10. Nested cross-validation  
11. Metric stability and confidence intervals  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import (
    train_test_split,
    KFold,
    StratifiedKFold,
    LeaveOneOut,
    cross_val_score,
    GridSearchCV,
)
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

np.random.seed(42)

## Create Example Datasets

We create:

- a regression dataset for continuous prediction tasks
- a classification dataset for stratified validation and permutation testing

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=220,
    n_features=8,
    n_informative=5,
    noise=15,
    random_state=42,
)

X_clf, y_clf = make_classification(
    n_samples=260,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.72, 0.28],
    random_state=42,
)

pd.Series(y_clf).value_counts().sort_index()

## 1. Train / Validation / Test Split

A common workflow is to divide the data into three parts:

- training set
- validation set
- test set

This supports model fitting, tuning, and final unbiased evaluation.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Size': [len(X_train), len(X_val), len(X_test)]
})

## 2. Holdout Validation

In holdout validation, the model is trained once and evaluated on one held-out set.

Holdout error estimate:

$$
\hat{E}_{holdout}=\frac{1}{n_{test}}\sum_{i\in test}L(y_i,\hat{y}_i)
$$

In [ ]:
holdout_model = LinearRegression()
holdout_model.fit(X_train, y_train)
y_val_pred = holdout_model.predict(X_val)
y_test_pred = holdout_model.predict(X_test)

pd.DataFrame({
    'Set': ['Validation', 'Test'],
    'RMSE': [
        mean_squared_error(y_val, y_val_pred) ** 0.5,
        mean_squared_error(y_test, y_test_pred) ** 0.5,
    ],
    'R2': [
        r2_score(y_val, y_val_pred),
        r2_score(y_test, y_test_pred),
    ]
})

## 3. k-Fold Cross-Validation

The k-fold estimate is:

$$
CV_k = \frac{1}{k}\sum_{j=1}^{k}E_j
$$

This is one of the most important standard validation approaches.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_r2 = cross_val_score(LinearRegression(), X_reg, y_reg, cv=kf, scoring='r2')
cv_scores_rmse = -cross_val_score(LinearRegression(), X_reg, y_reg, cv=kf, scoring='neg_root_mean_squared_error')

pd.DataFrame({
    'Fold': np.arange(1, 6),
    'R2': cv_scores_r2,
    'RMSE': cv_scores_rmse
})

In [ ]:
pd.DataFrame({
    'Metric': ['Mean R2', 'Std R2', 'Mean RMSE', 'Std RMSE'],
    'Value': [
        cv_scores_r2.mean(),
        cv_scores_r2.std(ddof=1),
        cv_scores_rmse.mean(),
        cv_scores_rmse.std(ddof=1),
    ]
})

## 4. Stratified Cross-Validation

When classes are imbalanced, stratified CV preserves class proportions in each fold.

This is especially important for classification tasks like defect detection or failure classification.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
acc_scores = cross_val_score(clf_model, X_clf, y_clf, cv=skf, scoring='accuracy')

pd.DataFrame({'Fold': np.arange(1, 6), 'Accuracy': acc_scores})

### Inspect Class Balance in One Stratified Split

In [ ]:
for fold_id, (train_idx, test_idx) in enumerate(skf.split(X_clf, y_clf), start=1):
    if fold_id == 1:
        train_counts = pd.Series(y_clf[train_idx]).value_counts(normalize=True).sort_index()
        test_counts = pd.Series(y_clf[test_idx]).value_counts(normalize=True).sort_index()
        balance_df = pd.DataFrame({'Train Proportion': train_counts, 'Test Proportion': test_counts})
        break
balance_df

## 5. Leave-One-Out Cross-Validation (LOOCV)

LOOCV is the special case where:

$$
k=n
$$

Each run leaves one sample out and trains on the rest.

In [ ]:
X_small = X_reg[:40]
y_small = y_reg[:40]
loo = LeaveOneOut()
loo_scores = cross_val_score(LinearRegression(), X_small, y_small, cv=loo, scoring='r2')

pd.DataFrame({
    'Metric': ['Mean LOOCV R2', 'Std LOOCV R2', 'Number of Runs'],
    'Value': [loo_scores.mean(), loo_scores.std(ddof=1), len(loo_scores)]
})

## 6. Repeated Random Subsampling

Also called Monte Carlo cross-validation. We repeatedly generate random train-test splits and aggregate the results.

Estimate:

$$
E = \frac{1}{R}\sum_{r=1}^{R}E_r
$$

In [ ]:
repeats = 30
rmse_list = []
r2_list = []

for r in range(repeats):
    X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.25, random_state=r)
    model = LinearRegression()
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    rmse_list.append(mean_squared_error(y_te, pred) ** 0.5)
    r2_list.append(r2_score(y_te, pred))

repeated_df = pd.DataFrame({'Repeat': np.arange(1, repeats + 1), 'RMSE': rmse_list, 'R2': r2_list})
repeated_df.head()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(repeated_df['Repeat'], repeated_df['RMSE'])
plt.title('Repeated Random Split RMSE Stability')
plt.xlabel('Repeat')
plt.ylabel('RMSE')
plt.show()

## 7. Bootstrap Resampling

Bootstrap draws samples with replacement.

Bootstrap variance estimate:

$$
Var_{boot}(\hat{\theta})=\frac{1}{B-1}\sum_{b=1}^{B}(\hat{\theta}^{*(b)}-\bar{\theta})^2
$$

In [ ]:
B = 200
boot_r2 = []

for b in range(B):
    X_boot, y_boot = resample(X_reg, y_reg, replace=True, random_state=1000 + b)
    model = LinearRegression()
    model.fit(X_boot, y_boot)
    pred = model.predict(X_test)
    boot_r2.append(r2_score(y_test, pred))

boot_r2 = np.array(boot_r2)
boot_mean = boot_r2.mean()
boot_std = boot_r2.std(ddof=1)
boot_ci = np.percentile(boot_r2, [2.5, 97.5])

pd.DataFrame({
    'Metric': ['Bootstrap Mean R2', 'Bootstrap Std R2', 'Bootstrap 95% CI Lower', 'Bootstrap 95% CI Upper'],
    'Value': [boot_mean, boot_std, boot_ci[0], boot_ci[1]]
})

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(boot_r2, bins=20)
plt.title('Bootstrap Distribution of R2')
plt.xlabel('R2')
plt.ylabel('Frequency')
plt.show()

## 8. Out-of-Bag (OOB) Intuition

In a bootstrap sample of size \(n\), about **36.8%** of observations are typically left out and can serve as validation data.

The probability that a given observation is not selected in one draw is:

$$
\left(1-\frac{1}{n}\right)^n \approx e^{-1} \approx 0.368
$$

In [ ]:
n_example = 500
oob_fraction_approx = (1 - 1 / n_example) ** n_example
oob_fraction_approx

## 9. Permutation Validation

Permutation testing checks whether the model signal is stronger than chance by shuffling labels.

A practical p-value estimate is:

$$
p=\frac{1+\#(M_{perm}\ge M_{real})}{1+B}
$$

for metrics where larger is better, such as accuracy.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf
)
perm_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
perm_model.fit(X_train_c, y_train_c)
real_acc = accuracy_score(y_test_c, perm_model.predict(X_test_c))

B_perm = 100
perm_acc = []
for b in range(B_perm):
    y_perm = np.random.permutation(y_train_c)
    perm_model.fit(X_train_c, y_perm)
    perm_acc.append(accuracy_score(y_test_c, perm_model.predict(X_test_c)))

perm_acc = np.array(perm_acc)
perm_p = (1 + np.sum(perm_acc >= real_acc)) / (1 + B_perm)

pd.DataFrame({
    'Metric': ['Real accuracy', 'Permutation mean accuracy', 'Permutation p-value'],
    'Value': [real_acc, perm_acc.mean(), perm_p]
})

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(perm_acc, bins=20)
plt.axvline(real_acc)
plt.title('Permutation Test Distribution of Accuracy')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.show()

## 10. Nested Cross-Validation

Nested CV uses:

- an inner loop for hyperparameter tuning
- an outer loop for unbiased evaluation

This is a gold-standard validation setup for publication-quality ML studies.

In [ ]:
outer_cv = KFold(n_splits=4, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

ridge_pipe = make_pipeline(StandardScaler(), Ridge())
param_grid = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
grid = GridSearchCV(ridge_pipe, param_grid=param_grid, cv=inner_cv, scoring='r2')

nested_scores = cross_val_score(grid, X_reg, y_reg, cv=outer_cv, scoring='r2')
pd.DataFrame({'Outer Fold': np.arange(1, len(nested_scores) + 1), 'Nested CV R2': nested_scores})

## 11. Metric Stability and Confidence Interval

Across repeated resampling, we often report:

$$
\bar{M}=\frac{1}{R}\sum_{r=1}^{R}M_r
$$

$$
s_M=\sqrt{\frac{1}{R-1}\sum (M_r-\bar{M})^2}
$$

$$
\bar{M}\pm 1.96\frac{s_M}{\sqrt{R}}
$$

In [ ]:
R = len(r2_list)
mean_metric = np.mean(r2_list)
std_metric = np.std(r2_list, ddof=1)
ci_low = mean_metric - 1.96 * std_metric / np.sqrt(R)
ci_high = mean_metric + 1.96 * std_metric / np.sqrt(R)

pd.DataFrame({
    'Metric': ['Mean R2', 'Std R2', '95% CI Lower', '95% CI Upper'],
    'Value': [mean_metric, std_metric, ci_low, ci_high]
})

## 12. Small Summary Table

This table gathers main validation outputs from the notebook.

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        '5-fold mean R2',
        '5-fold mean RMSE',
        'Stratified CV mean accuracy',
        'LOOCV mean R2',
        'Repeated split mean R2',
        'Bootstrap mean R2',
        'Permutation p-value',
        'Nested CV mean R2'
    ],
    'Value': [
        cv_scores_r2.mean(),
        cv_scores_rmse.mean(),
        acc_scores.mean(),
        loo_scores.mean(),
        np.mean(r2_list),
        boot_mean,
        perm_p,
        nested_scores.mean()
    ]
})
summary

## 13. Mini Exercises

Try these on your own:

1. Change the number of folds from 5 to 10 and compare the CV results.  
2. Increase the number of bootstrap replications and inspect the CI stability.  
3. Replace the linear model with ridge or random forest.  
4. Try repeated random subsampling on the classification dataset.  
5. Compare nested CV and ordinary CV for hyperparameter tuning.  
6. Replace the synthetic data with your own FEM, SHM, or surrogate dataset.

These exercises are especially useful for AI, machine learning, surrogate modeling, structural engineering, and scientific benchmarking.